### Using A Combination of Playwright and BeautifulSoup to Scrape All Parks and Overall Condition Scores

In [11]:
# !pip install playwright
# !playwright install

import requests
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time

Establish loop for each borogh page, to make sure they all work

In [12]:
boro_code = ["M", "X", "B", "Q", "R"]

headers = {'User-Agent': 'Mozilla/5.0 (Linux; Android 16; Pixel 10) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/153.0.0.0 Mobile Safari/537.36'}

for boro in boro_code:
    url = f"https://www.nycgovparks.org/park-features/parks-list?boro={boro}"
    response = requests.get(url, headers=headers)

    print(response.status_code)



200
200
200
200
200


And create a dictionary of page numbers for each borough page, to loop through.

In [13]:
content = BeautifulSoup(response.text, 'html.parser')
pagination_content = content.find_all("div", class_='pagination')

In [14]:
pages_by_boro = {}

for boro in boro_code:
    url = f"https://www.nycgovparks.org/park-features/parks-list?boro={boro}"
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # You'll need to confirm this selector against the page's actual HTML
    pagination_content = soup.select("div.pagination a")

    page_numbers = []
    for a in pagination_content:
        text = a.get_text(strip=True)
        if text.isdigit():          # skip "Next", "Prev", "..." etc.
            page_numbers.append(int(text))

    pages_by_boro[boro] = page_numbers
    print(boro, page_numbers)

M [1, 2, 3, 4, 31]
X [1, 2, 3, 4, 31]
B [1, 2, 3, 4, 50]
Q [1, 2, 3, 4, 50]
R [1, 2, 3, 4, 16]


In [15]:
boro_pages = {
    "M": list(range(1, 31 + 1)),
    "X": list(range(1, 31 + 1)),
    "B": list(range(1, 50 + 1)),
    "Q": list(range(1, 50 + 1)),
    "R": list(range(1, 16 + 1)),
}

Quick test to make sure it works:

In [16]:
print(boro_pages["X"])  # Example: prints the list of pages for Queens

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]


Now that we can successfully loop through each borough, we establish the rules for navigating through each page per borough. Then, we will navigate through each park name url, and grab the printed condition score.

In [17]:
session = requests.Session()
session.headers.update(headers)

r = session.get("https://www.nycgovparks.org/parks/bicentennial-veterans-memorial-park", timeout=30)
print(r.status_code)
print(r.text[:500])

200
<!doctype html>
<!--

                  `.--::::/:::--.`
            `.:/++++//:::::://++++/-.
         `-/+++:-.`            `.-:+++:.
       `:++/:.          `           .:++/-`
     `:++/.            -+`             -/++-
    ./+/-             .++/`              -++/`
   -++/`            -/+++++//             `/++.
  -++:            `-+++++++++:-             /++.
 `++/             `+++++++++++:             `/++`
 /++`   `:-...``` .+++++++++++- ....:/:::    .++:
`++:     -+++++++++++++++++++++


Scraper is commented out because it runs for approx. 6+ hours to complete:

In [18]:
""" 
import random

rows = []

boro_pages = {
    "M": list(range(1, 31 + 1)),
    "X": list(range(1, 31 + 1)),
    "B": list(range(1, 50 + 1)),
    "Q": list(range(1, 50 + 1)),
    "R": list(range(1, 16 + 1)),
}

session = requests.Session()
session.headers.update(headers)

# NEW: a reusable "get this URL, and if the site says no, wait and try again" function
def get_page(url):
    for attempt in range(3):                      # try up to 3 times
        r = session.get(url, timeout=30)
        if r.status_code == 200:                  # success: hand the page back
            return r
        print(f"Status {r.status_code}, sleeping 30 min: {url}")
        time.sleep(30 * 60)                       # blocked: wait 30 minutes, then loop and try again
    raise Exception(f"Still blocked after 3 tries: {url}")   # only stops if all 3 tries fail

park_count = 0   # NEW: counts parks scraped, used for the periodic break

# looping through the whole dictionary to load the url of each page for each borough
for boro, pages in boro_pages.items():
    for page in pages:
        url = f"https://www.nycgovparks.org/park-features/parks-list?boro={boro}&page={page}"
        response = get_page(url)   
        soup = BeautifulSoup(response.text, "html.parser")
        print(boro, page, len(soup.select("div.park-search-item p.more-left a[href]")))

        # then looping through each park name and park url to load each park page, to scrape its condition score
        for a in soup.select("div.park-search-item p.more-left a[href]"):
            park_name = a.get_text(strip=True)
            park_link = "https://www.nycgovparks.org" + a["href"]
            response_park = get_page(park_link)   
            soup_park = BeautifulSoup(response_park.text, "html.parser")

            score_tag = soup_park.select_one("div.park-score-number div.value")
            condition_score = score_tag.get_text(strip=True) if score_tag else None

            rows.append({"boro": boro, "park": park_name, "score": condition_score})
            time.sleep(random.uniform(2, 3.5))

            park_count += 1                                 
            if park_count % 100 == 0:                       
                pd.DataFrame(rows).to_csv("parks_checkpoint.csv", index=False)  # save a backup
                print("Checkpoint saved, pausing 5 min")
                time.sleep(5 * 60)                            # take a 5 minute break

df = pd.DataFrame(rows)
df["park"] = df["park"].str.replace(" Details", "", regex=False)
df = df.drop_duplicates().reset_index(drop=True)
df 
"""

' \nimport random\n\nrows = []\n\nboro_pages = {\n    "M": list(range(1, 31 + 1)),\n    "X": list(range(1, 31 + 1)),\n    "B": list(range(1, 50 + 1)),\n    "Q": list(range(1, 50 + 1)),\n    "R": list(range(1, 16 + 1)),\n}\n\nsession = requests.Session()\nsession.headers.update(headers)\n\n# NEW: a reusable "get this URL, and if the site says no, wait and try again" function\ndef get_page(url):\n    for attempt in range(3):                      # try up to 3 times\n        r = session.get(url, timeout=30)\n        if r.status_code == 200:                  # success: hand the page back\n            return r\n        print(f"Status {r.status_code}, sleeping 30 min: {url}")\n        time.sleep(30 * 60)                       # blocked: wait 30 minutes, then loop and try again\n    raise Exception(f"Still blocked after 3 tries: {url}")   # only stops if all 3 tries fail\n\npark_count = 0   # NEW: counts parks scraped, used for the periodic break\n\n# looping through the whole dictionary to l

In [19]:
# df.to_csv("../output/parks_condition_scores_FINAL.csv", index=False)

That scraper took over 6 hours to complete! This is because I had to code in multiple pauses, slow down the scraper, and added a 30 minute buffer for each rate limiting block. 

In [20]:
all_scores = pd.read_csv("../output/parks_condition_scores_FINAL.csv")

all_scores 

,boro,park,score
0,M,A Philip Randolph Square,100.0
1,M,ABC Playground,95.0
2,M,Abe Lebewohl Park,85.0
3,M,Abe Lebewohl Triangle,98.0
4,M,Abingdon Square,95.0
...,...,...,...
1640,R,Westerleigh Park,80.0
1641,R,Westwood Park,84.0
1642,R,Willowbrook Park,92.0
1643,R,Willowbrook Parkway,95.0


In [21]:
all_scores.dtypes

boro         str
park         str
score    float64
dtype: object

In [22]:
all_scores['score'].value_counts(dropna=False)

score
NaN      160
100.0    147
95.0      58
90.0      51
88.0      51
        ... 
42.0       2
32.0       1
30.0       1
45.0       1
44.0       1
Name: count, Length: 61, dtype: int64

In [23]:
len(all_scores)

1645

In [24]:
all_scores = all_scores.dropna(subset=["score"])

In [25]:
len(all_scores)

1485

In [26]:
all_scores.to_csv("../output/condition_scores_filtered.csv", index=False)